# Redrob Hackathon — Candidate Ranking Pipeline

**Goal:** Rank top 100 from 100K candidates for Senior AI Engineer.

**Runtime:** GPU (T4+) — *Runtime → Change runtime type → T4 GPU*

### Pipeline
1. Clone repo + extract dataset
2. Install deps
3. Pre-compute: bi-encoder embeddings + BM25 + cross-encoder re-ranking + features
4. Rank: XGBoost LTR → top 100 with reasoning
5. Validate + download

## Step 1 — GPU Check + Clone + Extract

In [ ]:
# Verify GPU
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError("No GPU! Go to Runtime → Change runtime type → T4 GPU")
print(result.stdout.split('\n')[0])
print("GPU OK")

In [ ]:
# Clone repo
!git clone https://github.com/JASHWANTHS07/india-runs.git
%cd india-runs

In [ ]:
# Extract dataset from Data.rar
!apt-get install -qq unrar
!unrar x -o+ dataset/Data.rar dataset/
print("\nExtraction complete.")
!ls dataset/India_runs_data_and_ai_challenge/

## Step 2 — Install Dependencies

In [ ]:
# Install deps (torch is pre-installed in Colab — don't reinstall)
!pip install -q sentence-transformers tqdm pandas pyarrow xgboost
print("Dependencies installed.")

In [ ]:
# Verify dataset
import os
CANDIDATES = "dataset/India_runs_data_and_ai_challenge/candidates.jsonl"
size_mb = os.path.getsize(CANDIDATES) / 1e6
print(f"candidates.jsonl: {size_mb:.1f} MB")
assert size_mb > 400, f"Too small ({size_mb:.1f} MB)"
print("Dataset OK")

## Step 3 — Pre-computation (GPU)

Runs three retrieval stages:
1. **Bi-encoder** — BAAI/bge-large-en-v1.5 embeddings for all 100K candidates
2. **BM25** — sparse keyword matching against JD
3. **Cross-encoder** — ms-marco-MiniLM-L-6-v2 re-ranks top-1000 by bi-encoder score

Plus 78 structured features (v2: headline, salary fit, assessment depth, market demand, template detection, career coherence).

**Expected time:** ~20–30 min on T4.

In [ ]:
!python src/precompute.py \
    --candidates dataset/India_runs_data_and_ai_challenge/candidates.jsonl \
    --artifacts  ./artifacts

In [ ]:
# Verify artifacts
import numpy as np
import pandas as pd

emb = np.load('artifacts/embeddings.npy')
jd  = np.load('artifacts/jd_embedding.npy')
df  = pd.read_parquet('artifacts/features.parquet')

assert emb.shape[1] == 1024, f"Unexpected dim: {emb.shape}"
assert 'bm25_score' in df.columns, "Missing bm25_score"
assert 'cross_encoder_score' in df.columns, "Missing cross_encoder_score"

print(f"embeddings.npy       : {emb.shape}  ({emb.nbytes/1e6:.0f} MB)")
print(f"jd_embedding         : {jd.shape}")
print(f"features.parquet     : {len(df)} rows, {len(df.columns)} cols")
print(f"  bm25 range         : [{df.bm25_score.min():.3f}, {df.bm25_score.max():.3f}]")
print(f"  cross-encoder range: [{df.cross_encoder_score.min():.3f}, {df.cross_encoder_score.max():.3f}]")
print("Artifacts OK")

## Step 4 — Rank (CPU, XGBoost LTR)

Multiplicative scoring hierarchy: retrieval depth as primary discriminator.
Trains XGBoost (rank:pairwise) on pseudo-relevance labels using 78 features
plus semantic, BM25, and cross-encoder scores. Sigmoid normalization for wider score spread.

**Expected time:** <3 min.

In [ ]:
!python src/rank.py \
    --artifacts ./artifacts \
    --out       ./jashwanth_s.csv \
    --method    xgboost \
    --tune      --tune-trials 50

## Step 5 — Validate & Download

In [ ]:
import pandas as pd, sys
from dataclasses import fields

df = pd.read_csv('jashwanth_s.csv')
assert len(df) == 100
assert set(df['rank']) == set(range(1, 101))
scores = df.sort_values('rank')['score'].tolist()
assert all(scores[i] >= scores[i+1] for i in range(len(scores)-1)), "Scores not non-increasing"

# Check tie-break: equal scores must have candidate_id ascending
rows = df.sort_values('rank')[['rank','score','candidate_id']].values.tolist()
for i in range(len(rows)-1):
    if rows[i][1] == rows[i+1][1]:
        assert rows[i][2] < rows[i+1][2], f"Tie-break violated at ranks {rows[i][0]},{rows[i+1][0]}"

# Honeypot check
sys.path.insert(0, '.')
from src.features import CandidateFeatures
from src.honeypot import is_honeypot
feat_df = pd.read_parquet('artifacts/features.parquet')
top_feats = feat_df[feat_df['candidate_id'].isin(set(df['candidate_id']))]
honeypots = []
for _, row in top_feats.iterrows():
    kwargs = {f.name: row[f.name] for f in fields(CandidateFeatures) if f.name in row.index}
    kwargs.setdefault('profile_text', '')
    if is_honeypot(CandidateFeatures(**kwargs)):
        honeypots.append(row['candidate_id'])

print(f"Rows      : {len(df)}")
print(f"Scores    : {scores[0]:.4f} → {scores[-1]:.4f}")
print(f"Honeypots : {len(honeypots)} (must be 0)")
assert len(honeypots) == 0
print("All checks passed.")

In [ ]:
# Top 10
pd.set_option('display.max_colwidth', 80)
df.sort_values('rank').head(10)[['rank', 'candidate_id', 'score', 'reasoning']]

In [ ]:
# Download
from google.colab import files
files.download('jashwanth_s.csv')
print("Downloaded.")